In [ ]:
%load_ext autoreload
%autoreload 2

Run Model

In [ ]:
# from peft import LoraConfig, get_peft_model
# from transformers import AutoModelForCausalLM
# import torch




# # model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
# model = AutoModelForCausalLM.from_pretrained('/root/MoRA/save_lora/checkpoint-1562')
# config = LoraConfig(
#     # enable MoRA
#     use_mora=True,
#     # type 1 (Sharing) for large lora ranks, Eq. 6 in paper
#     # type 6 (RoPE based) for small lora ranks, Eq. 9 in paper
#     mora_type=6,
#     # lora rank here, we will calculate corresponding $\hat{r}$ in MoRA
#     r=8,
#     # MoRA does not use lora_alpha
#     # lora_alpha=lora_alpha,
#     target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","down_proj","up_proj"],
#     # lora_dropout=lora_dropout,
#     task_type="CAUSAL_LM",
#     # **kwargs,
# )
# model = get_peft_model(model, config)

In [ ]:
# input_ids = torch.randint(0, 100, (1, 10)).to("cuda")
# outputs = model(input_ids)
# outputs.logits.shape
# # model.to("cuda")

Plot loss

In [ ]:

import re
import matplotlib.pyplot as plt

import re
import matplotlib.pyplot as plt

def plot_losses_from_logs(file_paths, labels=None):
    """
    Reads multiple training log files, extracts loss and epoch values, and plots them on the same curve.
    
    :param file_paths: List of paths to log files.
    :param labels: Optional list of labels corresponding to each log file.
    """
    if labels is None:
        labels = [f"Run {i+1}" for i in range(len(file_paths))]

    plt.figure(figsize=(8, 5))

    for file_path, label in zip(file_paths, labels):
        # Read the file
        with open(file_path, "r", encoding="utf-8") as file:
            log_content = file.read()

        # Regular expression to extract metric dictionaries
        metric_pattern = re.compile(r"\{'loss': ([\d.]+), 'grad_norm': [\d.eE+-]+, 'learning_rate': [\d.eE+-]+, 'epoch': ([\d.]+)\}")

        # Extract loss and epoch values
        losses = []
        epochs = []

        for match in metric_pattern.finditer(log_content):
            loss = float(match.group(1))
            epoch = float(match.group(2))
            losses.append(loss)
            epochs.append(epoch)

        if not losses:
            print(f"Warning: No loss data found in {file_path}")
            continue

        # Plot each loss curve
        plt.plot(epochs, losses, marker='o', linestyle='-', label=label)

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss Curves")
    plt.legend()
    plt.grid(True)
    plt.show()

# Example usage:
# plot_losses_from_logs(["log1.txt", "log2.txt"], labels=["Model A", "Model B"])


plot_losses_from_logs(['/root/MoRA/lora_run_8bit.log', '/root/MoRA/mora_run.log'], labels=['LoRA', 'MoRA'])



Data Setup Test

In [ ]:
import transformers
from training_utils import make_supervised_data_module
from collections import namedtuple

def setup_data_args(data_path, base_model, data_length=None, val_split=None, subset=None):
    DataArgs = namedtuple('DataArgs', ['data_path', 'data_length', 'val_split', 'subset', 'is_chat'])
    is_chat = 'Llama-3' in base_model and 'Instruct' in base_model
    return DataArgs(data_path, data_length, val_split, subset, is_chat)

def load_data(
    data_path: str,
    base_model: str,
    data_length: int = None,
    val_split: float = 0.0,
    subset: str = None,
) -> tuple:
    """
    Load and prepare dataset for training and evaluation.
    
    Args:
        data_path: Path to the dataset ('meta-math/MetaMathQA' or 'qiaojin/PubMedQA')
        base_model: Name of the base model to use
        data_length: Number of samples to use from dataset
        val_split: Fraction of data to use for validation
        model_max_length: Maximum sequence length for tokenizer. If None, uses defaults (512 for meta-math, 768 for PubMedQA)
    
    Returns:
        tuple: (train_dataset, eval_dataset, tokenizer)
    """
    # Setup data arguments based on dataset
    if 'meta-math' in data_path:
        model_max_length = 512
        data_args = setup_data_args(data_path, base_model, data_length=data_length, val_split=val_split)
    elif data_path == 'qiaojin/PubMedQA':
        model_max_length = 768
        data_args = setup_data_args(data_path, base_model, data_length=data_length, val_split=val_split, subset=subset)
    else:
        raise ValueError(f"Unsupported dataset: {data_path}")

    # Initialize tokenizer
    tokenizer = transformers.AutoTokenizer.from_pretrained(
        base_model,
        model_max_length=model_max_length,
        padding_side="right",
        use_fast=False,
    )
    tokenizer.pad_token_id = 2  # unk token, different from eos token

    # Create data module and return datasets
    data_module = make_supervised_data_module(tokenizer=tokenizer, data_args=data_args)
    return data_module, tokenizer

In [ ]:
# Define the parameters
# data_path = 'meta-math/MetaMathQA'
data_path = 'qiaojin/PubMedQA'

# Load the data using the new function
data_module, _ = load_data(
    data_path=data_path,
    base_model='meta-llama/Llama-3.2-1B-Instruct',
    data_length=100,
    val_split=0.2
)
train_data, eval_data = data_module['train_dataset'], data_module['eval_dataset'], tokenizer

In [ ]:
print(f'{len(eval_data)=}', f'{len(train_data)=}')
print(train_data[0].keys())

In [ ]:
# Get average length (in tokens) of the train_data
ravg = 0
for i in range(len(train_data)):
    seq = train_data[i]['input_ids'] + train_data[i]['labels']
    seq_tokens = tokenizer.encode(seq, add_special_tokens=False)
    # print(len(seq_tokens))
    ravg += len(seq_tokens)
print(ravg / len(train_data))


Pub Med Eval Data

In [ ]:

# Define the parameters
# data_path = 'meta-math/MetaMathQA'
data_path = 'qiaojin/PubMedQA'
subset = 'pqa_labeled'

# Load the data using the new function
data_module, tokenizer = load_data(
    data_path=data_path,
    base_model='meta-llama/Llama-3.2-1B-Instruct',
    subset=subset,
    # data_length=1000,
    # val_split=0.2
)
train_data, eval_data = data_module['train_dataset'], data_module['eval_dataset']

In [ ]:
# train_data.sources[0], train_data.targets[0]

Eval Pubmed Checkpoints

In [ ]:
from peft import AutoPeftModelForCausalLM
ckpt_path = '/root/MoRA/pub-med-qa/save_test_lora_rank128_lr1e-4/checkpoint-1200'
model = AutoPeftModelForCausalLM.from_pretrained(ckpt_path, device_map="auto")
model




===================================BUG REPORT===================================
Welcome to bitsandbytes. For bug reports, please submit your error trace to: https://github.com/TimDettmers/bitsandbytes/issues
CUDA_SETUP: WARNING! libcudart.so not found in any environmental path. Searching /usr/local/cuda/lib64...
CUDA SETUP: CUDA runtime path found: /usr/local/cuda/lib64/libcudart.so
CUDA SETUP: Highest compute capability among GPUs detected: 8.6
CUDA SETUP: Detected CUDA version 121
CUDA SETUP: Loading binary /opt/poetry-venv/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cuda121.so...


/opt/poetry-venv/lib/python3.10/site-packages/bitsandbytes/cuda_setup/main.py:136: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('tcp'), PosixPath('//10.152.183.1'), PosixPath('443')}
  warn(msg)
/opt/poetry-venv/lib/python3.10/site-packages/bitsandbytes/cuda_setup/main.py:136: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('8000'), PosixPath('tcp'), PosixPath('//10.152.183.117')}
  warn(msg)
/opt/poetry-venv/lib/python3.10/site-packages/bitsandbytes/cuda_setup/main.py:136: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('tcp'), PosixPath('//10.152.183.8'), PosixPath('8080')}
  warn(msg)
/opt/poetry-venv/lib/python3.10/site-packages/bitsandbytes/cuda_setup/main.py:136: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('8000'), PosixPath('tcp

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaSdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=128, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=128, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (k_proj): lora.Linear(
                (base_layer): Linear(in_featur

In [5]:
from training_utils import preprocess

data = preprocess(train_data.sources, train_data.targets, tokenizer)


In [6]:
# Calculate total number of samples and average sequence length
len(data['input_ids']), sum(len(x) for x in data['input_ids']) / len(data['input_ids'])

(1000, 427.456)

In [11]:
import torch
from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

# Convert to tensors on CPU first
input_ids = data['input_ids']
labels = data['labels']

# Pad sequences in batch
input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)

# Create attention mask
attention_mask = input_ids.ne(tokenizer.pad_token_id).long()

# Process in batches to avoid OOM
batch_size = 8
n_samples = input_ids.shape[0]
total_loss = 0

max_samples = 10 # 1000 samples for testing
n_samples = min(n_samples, max_samples)

with torch.no_grad():
    for i in tqdm(range(0, n_samples, batch_size)):
        # Move batch to GPU
        batch_input_ids = input_ids[i:i+batch_size].to(device)
        batch_attention_mask = attention_mask[i:i+batch_size].to(device)
        batch_labels = labels[i:i+batch_size].to(device)
        
        outputs = model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            labels=batch_labels
        )
        total_loss += outputs.loss.item() 
        
        # Clear GPU memory
        del batch_input_ids, batch_attention_mask, batch_labels
        torch.cuda.empty_cache()

avg_loss = total_loss / n_samples
print(f"Average Loss: {avg_loss:.4f}")


100%|██████████| 2/2 [00:02<00:00,  1.44s/it]

Average Loss: 0.3314


In [ ]:
data['labels'][0]